# Build a Master Dataset for Sport Objects

Import multiple source datasets into the master dataset and cache normalized
pixel digests.

In [1]:
%load_ext autoreload
%autoreload 2
%aimport -fiftyone -pathlib

from detection_common import configure_stdio_relative_path
from detection_common.utils.digest import image_content_digest
from detection_common.utils.json_io import print_json

from dataset_builder.config import DATA_ROOT, WORKSPACE_ROOT

# Display project paths relatively for consistent output across environments.
configure_stdio_relative_path(WORKSPACE_ROOT)

In [ ]:
from pathlib import Path

import fiftyone as fo
from fiftyone.types import COCODetectionDataset

# Name of target master dataset to be created/imported
MASTER_DATASET_NAME = "sports_objects_master"

In [ ]:
# REBUILD_MASTER_DATASET = True
REBUILD_MASTER_DATASET = False
if REBUILD_MASTER_DATASET and fo.dataset_exists(MASTER_DATASET_NAME):
    fo.delete_dataset(MASTER_DATASET_NAME)

In [4]:
def import_coco_split(
    dataset: fo.Dataset,
    split_dir: Path,
    label_path: Path,
    source_name: str,
    split: str,
) -> list[str]:
    """
    Merge a COCO split, cache pixel digests, and return newly added sample IDs.

    Preserve existing samples and fill missing digests after interrupted imports.
    """
    existing_ids = dataset.values("id")

    dataset.merge_dir(
        dataset_type=COCODetectionDataset,
        data_path=split_dir.as_posix(),
        labels_path=label_path.as_posix(),
        label_types=["detections"],
        label_field="original_labels",
        tags=[f"source:{source_name}"],
        key_field="filepath",
        skip_existing=True,
    )

    new_ids = dataset.exclude(existing_ids).values("id")
    if new_ids:
        dataset.set_values(
            "source_dataset",
            {sid: source_name for sid in new_ids},
            key_field="id",
        )
        dataset.set_values(
            "original_split",
            {sid: split for sid in new_ids},
            key_field="id",
        )

    unhashed_ids, unhashed_paths = dataset.exists("pixel_digest", False).values(
        ["id", "filepath"]
    )
    if unhashed_ids:
        with fo.ProgressBar(total=len(unhashed_ids)) as progress:
            digests = {
                sid: image_content_digest(path)
                for sid, path in progress(zip(unhashed_ids, unhashed_paths))
            }
        dataset.set_values("pixel_digest", digests, key_field="id")
    return new_ids

In [5]:
if fo.dataset_exists(MASTER_DATASET_NAME):
    master = fo.load_dataset(MASTER_DATASET_NAME)
else:
    master = fo.Dataset(MASTER_DATASET_NAME)
master.persistent = True
if not master.has_sample_field("pixel_digest"):
    master.add_sample_field("pixel_digest", fo.StringField)

Each sample's `pixel_digest` is the SHA-256 of its dimensions and RGBA pixels
after EXIF orientation normalization. Compute it once during import and reuse
it for duplicate detection. Recompute the field if the source media changes.
Run `nb01.07.02` after importing to tag exact pixel duplicates for review.

## Import gaelic football dataset

Preserve provenance: `source_dataset` records the original dataset name, and
`original_split` records "train", "val"/"valid", or "test" when available.

In [6]:
splits = ("train", "val", "valid", "test")
source_dir = DATA_ROOT / "composed" / "coco_gaelic_football"

for split in splits:
    split_dir = source_dir / "images" / split
    label_path = source_dir / "annotations" / f"instances_{split}.json"

    if not label_path.is_file():
        continue

    import_coco_split(
        dataset=master,
        split_dir=split_dir,
        label_path=label_path,
        source_name=source_dir.name,
        split=split,
    )

print("Total images:", master.count())
print_json(master.count_sample_tags())

 100% |███████████████████| 86/86 [56.0ms elapsed, 0s remaining, 1.5K samples/s]   
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.07.130226'
Converting existing index 'filepath' to unique on dataset 'sports_objects_master'
 100% |███████████████████| 86/86 [1.1s elapsed, 0s remaining, 80.3 samples/s]         
 100% |███████████████████| 20/20 [24.2ms elapsed, 0s remaining, 825.5 samples/s]     
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.08.364424'
 100% |███████████████████| 20/20 [108.1ms elapsed, 0s remaining, 185.0 samples/s]    
 100% |███████████████████| 18/18 [23.5ms elapsed, 0s remaining, 766.2 samples/s]     
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.08.576231'
 100% |███████████████████| 18/18 [61.6ms elapsed, 0s remaining, 292.2 samples/s] 
Total images: 124
{
  "source:coco_gaelic_football": 124
}


## Import Roboflow source datasets

Discover COCO exports under `<dataset>/<version>/<split>`. Preserve the
source dataset name and original split, and skip existing image filepaths.

In [7]:
roboflow_root = DATA_ROOT / "sources" / "roboflow"

if not roboflow_root.is_dir():
    raise FileNotFoundError(f"Roboflow source directory not found: {roboflow_root}")

for label_path in sorted(roboflow_root.glob("*/*/*/_annotations.coco.json")):
    split_dir = label_path.parent
    split = split_dir.name
    if split not in splits:
        continue

    source_dir = split_dir.parent.parent
    new_ids = import_coco_split(
        dataset=master,
        split_dir=split_dir,
        label_path=label_path,
        source_name=source_dir.name,
        split=split,
    )
    print(f"{source_dir.name}/{split_dir.parent.name}/{split}: {len(new_ids)} added")

 100% |█████████████████████| 1/1 [13.4ms elapsed, 0s remaining, 79.1 samples/s] 
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.08.774176'
 100% |█████████████████████| 1/1 [2.2ms elapsed, 0s remaining, 454.8 samples/s] 
ball-tracker-jpjs8/4/test: 1 added
 100% |█████████████████| 808/808 [426.6ms elapsed, 0s remaining, 1.9K samples/s]      
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.08.943335'
 100% |█████████████████| 808/808 [1.3s elapsed, 0s remaining, 631.8 samples/s]         
ball-tracker-jpjs8/4/train: 808 added
 100% |███████████████████| 48/48 [54.4ms elapsed, 0s remaining, 882.3 samples/s]     
Converting existing index 'filepath' to unique on dataset '2026.10.29.11.06.10.902814'
 100% |███████████████████| 48/48 [178.3ms elapsed, 0s remaining, 269.1 samples/s]     
basketball-lokg2/1/test: 48 added
 100% |█████████████████| 831/831 [673.0ms elapsed, 0s remaining, 1.2K samples/s]      
Converting existing index 'f

In [8]:
print("Total images:", master.count())
print_json(master.count_sample_tags())

Total images: 34003
{
  "source:ball-tracker-jpjs8": 809,
  "source:basketball-lokg2": 976,
  "source:basketball-xil7x": 9515,
  "source:coco_gaelic_football": 124,
  "source:football-2k4ty": 198,
  "source:football-5beob": 28,
  "source:football-ablqb": 100,
  "source:football-csdy0": 195,
  "source:football-detection-ftt4q": 806,
  "source:football-detection-gjrtq": 600,
  "source:football-gsgma": 879,
  "source:football-rmhvy": 1499,
  "source:football-tnb17": 240,
  "source:football-wsef2": 168,
  "source:football-ywyvk": 339,
  "source:tennis-racket-r6mgq": 16607,
  "source:tracer-football": 920
}
